**Q1**: Consider GPT-2 XL, which has the following configuration:

In [12]:
d_model = 1600
num_heads = 25
d_ff = 6400
num_layers = 48
vocab_size = 50257
context_length = 1024

Suppose we constructed our model using this configuration. How many trainable parameters
would our model have? Assuming each parameter is represented using single-precision floating
point, how much memory is required to just load this model?

In [26]:
def memory_analysis(d_model, num_heads, d_ff, num_layers, vocab_size, context_length):
    # Embedding matrix 
    emb_params = d_model * vocab_size

    # Transformer Block
    # RMSNorm: gain
    ln_params = d_model
    # MHA: WQ, WK, WV, WO 
    mha_params = 4 * d_model * d_model
    # SwiGLU: W1, W3 d_model -> d_ff, W2: d_ff -> d_model
    ff_params = 3 * d_ff * d_model
    tf_block_params = ln_params * 2 + mha_params + ff_params

    # Final norm block: gain
    final_ln_params = ln_params
    # Final linear block: d_model -> vocab_size
    final_linear_params = vocab_size * d_model

    total_gpt2_params = emb_params + num_layers * tf_block_params + final_ln_params + final_linear_params
    total_gpt2_memory = total_gpt2_params * 4

    print(rf"""Total Parameters: {total_gpt2_params}
Memory needed: {total_gpt2_memory / 1024 / 1024:.2f}MiB""")

memory_analysis(d_model, num_heads, d_ff, num_layers, vocab_size, context_length)

Total Parameters: 2127057600
Memory needed: 8114.08MiB


**Q2**: Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped
model. How many FLOPs do these matrix multiplies require in total? Assume that our input
sequence has `context_length` tokens.

In [40]:
def mm_flops_analysis(d_model, num_heads, d_ff, num_layers, vocab_size, context_length):
    block_flops = 0
    # In each transformer block, we need to compute Q, K, V, O
    # each mm is (d_model, d_model) @ (d_model, tokens) for each token in the sequence.
    # mm for (a, b) @ (b, c) shape needs flops of 2 * a * b * c (one for addition and one for multiplication)
    qkvo_flops = 4 * 2 * d_model * d_model * context_length
    block_flops += qkvo_flops
    # computing attention score QK^T (tokens, d_k) @ (d_k, tokens) for each head. Note that we use
    # d_model here because d_model = num_heads * d_k
    # plus weighted sum: mm attention with value (tokens, tokens) @ (tokens, d_model)
    attention_flops = 2 * context_length * context_length * d_model + 2 * context_length * context_length * d_model
    block_flops += attention_flops

    # SwiGLU
    # 2 mm for (tokens, d_model) @ (d_model, d_ff). 1 mm for (tokens, d_ff) @ (d_ff, d_model)
    swiglu_flops = 3 * 2 * (context_length * d_model * d_ff)
    block_flops += swiglu_flops

    # Final Linear
    # map d_model to vocab_size: (tokens, d_model) @ (d_model, vocab_size)
    final_linear_flops = 2 * context_length * d_model * vocab_size

    total_flops = block_flops * num_layers + final_linear_flops
    total_qkvo_flops = qkvo_flops * num_layers
    total_attention_flops = attention_flops * num_layers
    total_swiglu_flops = swiglu_flops * num_layers
    print(f"Total Flops: {total_flops:.3e}")
    print(f"QKVO: {total_qkvo_flops / total_flops:.2f} -> {total_qkvo_flops:.3e}")
    print(f"Attention: {total_attention_flops / total_flops:.2f} -> {total_attention_flops:.3e}")
    print(f"SwiGLU: {total_swiglu_flops / total_flops:.2f} -> {total_swiglu_flops:.3e}")
    print(f"Final Linear: {final_linear_flops / total_flops:.2f} -> {final_linear_flops:.3e}")

print("GPT-2 XL")
mm_flops_analysis(d_model, num_heads, d_ff, num_layers, vocab_size, context_length)

GPT-2 XL
Total Flops: 4.513e+12
QKVO: 0.22 -> 1.007e+12
Attention: 0.07 -> 3.221e+11
SwiGLU: 0.67 -> 3.020e+12
Final Linear: 0.04 -> 1.647e+11


**Q3**: Based on your analysis above, which parts of the model require the most FLOPs?

In [41]:
print("SwiGLU")

SwiGLU


**Q4**: Repeat your analysis with GPT-2 small (12 layers, 768 d_model, 12 heads), GPT-2 medium (24 layers, 1024 d_model, 16 heads), and GPT-2 large (36 layers, 1280 d_model, 20 heads). As the model size increases, which parts of the Transformer LM take up proportionally more or less of the total FLOPs?

In [43]:
print("GPT-2 small")
memory_analysis(768, 12, 6400, 12, 50257, 1024)
mm_flops_analysis(768, 12, 6400, 12, 50257, 1024)
print("=========")
print("GPT-2 Medium")
memory_analysis(1024, 16, 6400, 24, 50257, 1024)
mm_flops_analysis(1024, 16, 6400, 24, 50257, 1024)
print("=========")
print("GPT-2 Large")
memory_analysis(1280, 36, 6400, 36, 50257, 1024)
mm_flops_analysis(1280, 36, 6400, 36, 50257, 1024)

GPT-2 small
Total Parameters: 282472704
Memory needed: 1077.55MiB
Total Flops: 5.381e+11
QKVO: 0.11 -> 5.798e+10
Attention: 0.07 -> 3.865e+10
SwiGLU: 0.67 -> 3.624e+11
Final Linear: 0.15 -> 7.905e+10
GPT-2 Medium
Total Parameters: 675499008
Memory needed: 2576.82MiB
Total Flops: 1.381e+12
QKVO: 0.15 -> 2.062e+11
Attention: 0.07 -> 1.031e+11
SwiGLU: 0.70 -> 9.664e+11
Final Linear: 0.08 -> 1.054e+11
GPT-2 Large
Total Parameters: 1249416960
Memory needed: 4766.15MiB
Total Flops: 2.620e+12
QKVO: 0.18 -> 4.832e+11
Attention: 0.07 -> 1.933e+11
SwiGLU: 0.69 -> 1.812e+12
Final Linear: 0.05 -> 1.317e+11


**Q5**: Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one
forward pass change? How do the relative contribution of FLOPs of the model components
change?

In [44]:
mm_flops_analysis(d_model, num_heads, d_ff, num_layers, vocab_size, 16384)

Total Flops: 1.495e+14
QKVO: 0.11 -> 1.611e+13
Attention: 0.55 -> 8.246e+13
SwiGLU: 0.32 -> 4.832e+13
Final Linear: 0.02 -> 2.635e+12
